# Cuaderno 1 — Descarga de datos

**Tesis:** Predicción del Punto de Equilibrio de Flujo de Caja en Empresas de Crecimiento  
**Autores:** Andrea Meneses · Juan Ramos  
**Universidad EAFIT · 2026**

---

## Objetivo

Construir el dataset histórico de variables financieras trimestrales para las empresas del universo de estudio. Los datos se descargan desde dos fuentes:

- **S&P 500** (GitHub): lista actualizada de empresas con sus sectores
- **SEC EDGAR** (API pública del gobierno americano): estados financieros históricos desde 2007

## Universo de estudio

- Empresas del S&P 500 listadas en NYSE o NASDAQ
- Se excluyen cuatro sectores que distorsionan el FCF operativo:
  - **Financials**: bancos y aseguradoras — su FCF no se interpreta igual al de empresas operativas
  - **Utilities**: monopolios regulados — casi nunca tienen FCF negativo, no aplica el modelo
  - **Real Estate**: el FCF depende de compraventas de propiedades, no de la operación
  - **Energy**: el FCF depende del precio del petróleo, no de la salud del negocio

## Variables descargadas

Se descargan 20 variables financieras trimestrales (formularios 10-Q) organizadas en tres estados financieros:

| Grupo | Variables |
|---|---|
| Flujo de caja | FCF operativo, CAPEX, depreciación, stock-based compensation |
| Balance general | Caja, activos totales, activos corrientes, pasivos corrientes, deuda LP, inventario, cuentas por cobrar/pagar |
| Estado de resultados | Ingresos, costo de ventas, utilidad bruta, gastos I+D, gastos operativos, utilidad operativa, utilidad neta, EBITDA proxy |

> **Nota:** `cambio_capital_trabajo` no está disponible directamente en la SEC. Se calcula en el siguiente cuaderno como `activos_corrientes - pasivos_corrientes`.

## 1. Configuración inicial

Cargamos las librerías y las credenciales desde el archivo `.env`. La SEC EDGAR requiere un `User-Agent` con nombre y correo para identificar al consumidor de la API — es su política de uso responsable.

Para configurar el `.env`, crea un archivo llamado `.env` en la raíz del proyecto con el siguiente contenido:
```
SEC_USER_AGENT=Tu Nombre tuemail@email.com
```

In [4]:
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv
from pathlib import Path

# Cargar variables de entorno
load_dotenv()

# Credenciales para SEC EDGAR
SEC_USER_AGENT = os.getenv('SEC_USER_AGENT')
headers = {'User-Agent': SEC_USER_AGENT}

print('User-Agent configurado exitosamente')

User-Agent configurado exitosamente


## 2. Universo de empresas

Descargamos la lista actualizada del S&P 500 desde GitHub y aplicamos los filtros de sector.

In [5]:
# Descargar lista del S&P 500
url_sp500 = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
sp500 = pd.read_csv(url_sp500)

print(f'Total empresas S&P 500: {len(sp500)}')
print('\nEmpresas por sector:')
print(sp500['GICS Sector'].value_counts())

Total empresas S&P 500: 503

Empresas por sector:
GICS Sector
Industrials               79
Financials                76
Information Technology    71
Health Care               60
Consumer Discretionary    48
Consumer Staples          36
Utilities                 31
Real Estate               31
Materials                 26
Communication Services    23
Energy                    22
Name: count, dtype: int64


In [9]:
# Aplicar filtros de sector
sectores_excluidos = ['Financials', 'Utilities',
                      'Real Estate', 'Energy']
universo = sp500[~sp500['GICS Sector'].isin(sectores_excluidos)].copy()

print(f'Empresas después del filtro: {len(universo)}')
print('\nEmpresas por sector:')
universo['GICS Sector'].value_counts().reset_index()

Empresas después del filtro: 343

Empresas por sector:


,GICS Sector,count
0,Industrials,79
1,Information Technology,71
2,Health Care,60
3,Consumer Discretionary,48
4,Consumer Staples,36
5,Materials,26
6,Communication Services,23


In [10]:
# Mostrar la participación de cada sector
print('Participación de cada sector:')
universo['GICS Sector'].value_counts(normalize=True).reset_index()

Participación de cada sector:


,GICS Sector,proportion
0,Industrials,0.230321
1,Information Technology,0.206997
2,Health Care,0.174927
3,Consumer Discretionary,0.139942
4,Consumer Staples,0.104956
5,Materials,0.075802
6,Communication Services,0.067055


## 3. Obtener CIKs de la SEC

Cada empresa tiene un identificador único en la SEC llamado **CIK** (*Central Index Key*). Lo necesitamos para consultar sus estados financieros. La SEC publica un archivo con todos los CIKs que cruzamos contra nuestro universo.

In [11]:
# Descargar todos los CIKs de la SEC
url_ciks = "https://www.sec.gov/files/company_tickers.json"
respuesta = requests.get(url_ciks, headers=headers)
todos_ciks = pd.DataFrame(respuesta.json()).T

print(f"Total empresas en SEC: {len(todos_ciks)}")
todos_ciks.head()

Total empresas en SEC: 10462


,cik_str,ticker,title
0,1045810,NVDA,NVIDIA CORP
1,1652044,GOOGL,Alphabet Inc.
2,320193,AAPL,Apple Inc.
3,789019,MSFT,MICROSOFT CORP
4,1018724,AMZN,AMAZON COM INC


In [12]:
# Cruzar universo con CIKs
universo_con_cik = universo.merge(
    todos_ciks[["ticker", "cik_str"]],
    left_on="Symbol",
    right_on="ticker",
    how="left"
)

# Corrección manual: Brown-Forman usa BF-B en la SEC (no BF.B)
universo_con_cik.loc[universo_con_cik["Symbol"] == "BF.B", "cik_str"] = "14693"

con_cik = universo_con_cik["cik_str"].notna().sum()
sin_cik = universo_con_cik["cik_str"].isna().sum()

print(f"Empresas con CIK encontrado: {con_cik}")
print(f"Empresas sin CIK: {sin_cik}")

Empresas con CIK encontrado: 343
Empresas sin CIK: 0


## 4. Descarga de variables financieras desde SEC EDGAR

Descargamos 20 variables financieras para cada empresa directamente desde la API de SEC EDGAR. Por cada empresa hacemos una sola petición HTTP que trae todos sus reportes históricos, y de ahí extraemos los conceptos que nos interesan.

Solo conservamos registros del formulario **10-Q** (trimestral).

> **Tiempo estimado:** 10–15 minutos para las 343 empresas.

In [13]:
# Variables a descargar — nombre local : concepto en SEC EDGAR
variables_sec = {
    # Flujo de caja
    "fcf_operativo":      "NetCashProvidedByUsedInOperatingActivities",
    "capex":              "PaymentsToAcquirePropertyPlantAndEquipment",
    "depreciacion":       "DepreciationDepletionAndAmortization",
    "stock_based_comp":   "AllocatedShareBasedCompensationExpense",

    # Balance general
    "caja":               "CashAndCashEquivalentsAtCarryingValue",
    "activos_totales":    "Assets",
    "activos_corrientes": "AssetsCurrent",
    "pasivos_corrientes": "LiabilitiesCurrent",
    "deuda_largo_plazo":  "LongTermDebt",
    "inventario":         "InventoryNet",
    "cuentas_por_cobrar": "AccountsReceivableNetCurrent",
    "cuentas_por_pagar":  "AccountsPayableCurrent",

    # Estado de resultados
    "ingresos":           "Revenues",
    "costo_ventas":       "CostOfGoodsAndServicesSold",
    "utilidad_bruta":     "GrossProfit",
    "gastos_id":          "ResearchAndDevelopmentExpense",
    "gastos_operativos":  "OperatingExpenses",
    "utilidad_operativa": "OperatingIncomeLoss",
    "utilidad_neta":      "NetIncomeLoss",
    "ebitda_proxy":       "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
}

print(f"Variables a descargar: {len(variables_sec)}")

Variables a descargar: 20


In [16]:
def extraer_variable(us_gaap, nombre, concepto, ticker, sector):
    """Extrae una variable financiera del JSON de la SEC para una empresa."""
    if concepto not in us_gaap:
        return None

    registros = us_gaap[concepto]["units"].get("USD", [])
    if not registros:
        return None

    df = pd.DataFrame(registros)
    df = df[df["form"] == "10-Q"].copy()

    if df.empty:
        return None

    df["ticker"] = ticker
    df["sector"] = sector
    df["variable"] = nombre
    return df


# Descarga principal
todos_datos = []
errores = []

for i, fila in universo_con_cik.iterrows():
    ticker = fila["Symbol"]
    cik = str(fila["cik_str"]).zfill(10)
    sector = fila["GICS Sector"]

    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

    try:
        respuesta = requests.get(url, headers=headers, timeout=30)

        if respuesta.status_code != 200:
            errores.append(ticker)
            continue

        us_gaap = respuesta.json().get("facts", {}).get("us-gaap", {})

        for nombre, concepto in variables_sec.items():
            df_var = extraer_variable(
                us_gaap, nombre, concepto, ticker, sector)
            if df_var is not None:
                todos_datos.append(df_var)

    except Exception as e:
        errores.append(ticker)

    if (i + 1) % 50 == 0:
        print(f"{i + 1}/343 empresas procesadas...")

    time.sleep(0.3)

# Consolidar
df_variables = pd.concat(todos_datos, ignore_index=True)

print(f"\nDescarga completa.")
print(f"Shape: {df_variables.shape}")
print(f"Empresas: {df_variables['ticker'].nunique()}")
print(f"Variables: {df_variables['variable'].nunique()}")
print(f"Desde: {df_variables['end'].min()}")
print(f"Hasta: {df_variables['end'].max()}")

if errores:
    print(f"\nEmpresas con error ({len(errores)}): {errores}")

50/343 empresas procesadas...
100/343 empresas procesadas...
150/343 empresas procesadas...
200/343 empresas procesadas...
250/343 empresas procesadas...
300/343 empresas procesadas...

Descarga completa.
Shape: (513807, 12)
Empresas: 343
Variables: 20
Desde: 2007-06-30
Hasta: 2026-02-28


## 5. Guardar datos

Guardamos el dataset crudo en `datos/crudos/` para no tener que volver a descargarlo. Este archivo es el punto de partida del siguiente cuaderno donde construiremos las variables del modelo.

In [17]:
# Guardar dataset crudo
ruta = "../datos/crudos/variables_financieras_sec.csv"
df_variables.to_csv(ruta, index=False)

# Guardar también el universo con CIKs para uso posterior
universo_con_cik.to_csv("../datos/crudos/universo_empresas.csv", index=False)

# Verificar
tamano_mb = Path(ruta).stat().st_size / 1024
print(f"Guardado en: {ruta}")
print(f"Tamaño: {tamano_mb:.0f} KB")
print(f"\nRegistros por variable:")
df_variables["variable"].value_counts().reset_index()

Guardado en: ../datos/crudos/variables_financieras_sec.csv
Tamaño: 57914 KB

Registros por variable:


,variable,count
0,utilidad_neta,47459
1,caja,46750
2,utilidad_operativa,43748
3,activos_totales,31477
4,activos_corrientes,30054
5,pasivos_corrientes,30017
6,fcf_operativo,27016
7,ebitda_proxy,26226
8,utilidad_bruta,24964
9,capex,24457
